# PHÂN TÍCH CẢM XÚC ĐÁNH GIÁ SHOPEE TIẾNG VIỆT (SENTIMENT ANALYSIS)

Dự án cuối kỳ môn **Thực hành Xử lý Ngôn ngữ Tự nhiên (NLP)**
Sinh viên thực hiện: **Nguyễn Phúc Bách** - MSSV: **23280039**

Sổ tay (notebook) này được thiết kế để chạy mượt mà trên cả máy cá nhân (Local) và **Google Colab** (hỗ trợ liên kết Google Drive để lưu trữ tự động các mô hình đã huấn luyện).

In [ ]:
# --- CẤU HÌNH MOUNT GOOGLE DRIVE NẾU CHẠY TRÊN COLAB ---
import os
import sys

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Đang chạy trên Google Colab. Tiến hành mount Google Drive...")
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    
    # Thư mục lưu trữ kết quả trên Google Drive của bạn
    DRIVE_PATH = "/content/drive/MyDrive/shopee-reviews-analysis"
    os.makedirs(DRIVE_PATH, exist_ok=True)
    print(f"Đã đảm bảo thư mục lưu trữ tồn tại trên Drive: {DRIVE_PATH}")
    
    # Kiểm tra xem các file dữ liệu và code gốc đã có sẵn trong Colab chưa
    if not os.path.exists("shopee_reviews_dataset.jsonl") or not os.path.exists("utils.py"):
        print("Không tìm thấy file code hoặc dữ liệu gốc trong session. Đang tiến hành tải từ GitHub...")
        !git clone https://github.com/BrooksNguyen/shopee-reviews-analysis.git
        os.chdir("/content/shopee-reviews-analysis")
        
    PROJECT_PATH = os.getcwd()
    OUTPUT_PATH = DRIVE_PATH
    print(f"Thư mục chạy code: {PROJECT_PATH}")
    print(f"Thư mục lưu mô hình (Google Drive): {OUTPUT_PATH}")
    
    # Tự động cài đặt các thư viện cần thiết trên Colab
    print("Đang cài đặt các thư viện bổ sung (underthesea, transformers)...")
    !pip install -q underthesea transformers tqdm
    
    sys.path.append(PROJECT_PATH)
else:
    print("Đang chạy trên môi trường Local.")
    PROJECT_PATH = "."
    OUTPUT_PATH = "."


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import time
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from tqdm.notebook import tqdm

tqdm.pandas()

# Import bộ tiền xử lý custom
from utils import preprocess_review
print("Import thành công các thư viện cần thiết!")


## 1. Đọc và Chia Tách Dữ Liệu

In [ ]:
dataset_file = os.path.join(PROJECT_PATH, "shopee_reviews_dataset.jsonl")
print(f"Đọc dữ liệu từ: {dataset_file}")

df_raw = pd.read_json(dataset_file, lines=True)
print(f"Đọc thành công {len(df_raw)} dòng bình luận.")

# Chuẩn hóa nhãn và chọn cột
df_raw['sentiment'] = df_raw['label'].map({'positive': 1, 'negative': 0})
df = df_raw[['review', 'sentiment']].rename(columns={'review': 'comment'}).copy()

# Chia tập dữ liệu: 80% Train / 10% Val / 10% Test (Stratified)
df_train_full, df_test = train_test_split(df, test_size=0.1, random_state=42, stratify=df['sentiment'])
df_train, df_val = train_test_split(df_train_full, test_size=0.1111, random_state=42, stratify=df_train_full['sentiment'])

print(f"Số lượng Train: {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}")


## 2. Tiền Xử Lý Dữ Liệu & Phân Tích Độ Dài Câu (EDA)

In [ ]:
print("Đang tiền xử lý toàn bộ bình luận (làm sạch teencode, lặp ký tự, tách từ)...")
df_train['clean_comment'] = df_train['comment'].progress_apply(preprocess_review)
df_val['clean_comment'] = df_val['comment'].progress_apply(preprocess_review)
df_test['clean_comment'] = df_test['comment'].progress_apply(preprocess_review)

# Tính độ dài câu (đếm số từ)
df_train['len_raw'] = df_train['comment'].apply(lambda x: len(str(x).split()))
df_train['len_clean'] = df_train['clean_comment'].apply(lambda x: len(str(x).split()))

print("\n--- THỐNG KÊ ĐỘ DÀI BÌNH LUẬN (TẬP TRAIN) ---")
print(f"Độ dài trung bình trước tiền xử lý: {df_train['len_raw'].mean():.2f} từ")
print(f"Độ dài trung bình sau tiền xử lý:  {df_train['len_clean'].mean():.2f} từ")
print(f"Độ dài tối đa: {df_train['len_clean'].max()} từ")
print(f"Độ dài tối thiểu: {df_train['len_clean'].min()} từ")

# Loại bỏ dòng trống và dòng quá ngắn (dưới 2 từ sau khi xử lý)
df_train = df_train[df_train['len_clean'] >= 2].dropna()
df_val = df_val[df_val['clean_comment'].apply(lambda x: len(str(x).split())) >= 2].dropna()
df_test = df_test[df_test['clean_comment'].apply(lambda x: len(str(x).split())) >= 2].dropna()

print(f"\nSố mẫu tập Train sau khi lọc câu ngắn: {len(df_train)}")


In [ ]:
# Trực quan hóa dữ liệu
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.countplot(x='sentiment', data=df_train, palette='Set2')
plt.title("Phân Phối Nhãn Cảm Xúc (0: Tiêu cực, 1: Tích cực)")
plt.xlabel("Nhãn")
plt.ylabel("Số lượng")

plt.subplot(1, 2, 2)
sns.histplot(df_train['len_clean'], bins=40, color='teal', kde=True)
plt.title("Phân Phối Số Từ Trong Đánh Giá")
plt.xlabel("Số lượng từ")
plt.ylabel("Tần suất")

plt.tight_layout()
plt.show()


## 3. Huấn Luyện Các Mô Hình Machine Learning Truyền Thống (Baseline)
Sử dụng biểu diễn TF-IDF kết hợp với thuật toán Naive Bayes và Logistic Regression.

In [ ]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=10000)
X_train_tfidf = vectorizer.fit_transform(df_train['clean_comment'])
X_test_tfidf = vectorizer.transform(df_test['clean_comment'])

y_train = df_train['sentiment'].values
y_test = df_test['sentiment'].values

# 1. Naive Bayes
t0 = time.time()
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
nb_time = time.time() - t0
y_pred_nb = nb_model.predict(X_test_tfidf)
nb_acc = accuracy_score(y_test, y_pred_nb)
nb_f1 = f1_score(y_test, y_pred_nb, average='macro')

# 2. Logistic Regression
t0 = time.time()
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_tfidf, y_train)
lr_time = time.time() - t0
y_pred_lr = lr_model.predict(X_test_tfidf)
lr_acc = accuracy_score(y_test, y_pred_lr)
lr_f1 = f1_score(y_test, y_pred_lr, average='macro')

# Lưu model và vectorizer ra thư mục lưu trữ (Google Drive trên Colab hoặc thư mục hiện tại local)
with open(os.path.join(OUTPUT_PATH, 'nb_model.pkl'), 'wb') as f:
    pickle.dump(nb_model, f)
with open(os.path.join(OUTPUT_PATH, 'lr_model.pkl'), 'wb') as f:
    pickle.dump(lr_model, f)
with open(os.path.join(OUTPUT_PATH, 'tfidf_vectorizer.pkl'), 'wb') as f:
    pickle.dump(vectorizer, f)

print(f"Naive Bayes -> Acc: {nb_acc*100:.2f}%, F1: {nb_f1*100:.2f}% (Train time: {nb_time:.4f}s)")
print(f"Logistic Regression -> Acc: {lr_acc*100:.2f}%, F1: {lr_f1*100:.2f}% (Train time: {lr_time:.4f}s)")


## 4. Huấn Luyện Mô Hình Deep Learning (PyTorch Bi-LSTM)

In [ ]:
from collections import Counter

# 1. Xây dựng bộ từ điển (Vocab)
all_words = []
for text in df_train['clean_comment']:
    all_words.extend(text.split())

vocab_counter = Counter(all_words)
filtered_words = [word for word, count in vocab_counter.items() if count >= 2]
vocab = {word: idx + 2 for idx, word in enumerate(filtered_words)}
vocab['<PAD>'] = 0
vocab['<UNK>'] = 1

with open(os.path.join(OUTPUT_PATH, 'vocab.json'), 'w', encoding='utf-8') as f:
    json.dump(vocab, f, ensure_ascii=False, indent=2)

max_len_lstm = 80

class ReviewDataset(Dataset):
    def __init__(self, df, vocab, max_len):
        self.labels = df['sentiment'].values
        self.sequences = []
        for text in df['clean_comment']:
            seq = [vocab.get(w, vocab['<UNK>']) for w in text.split()]
            if len(seq) < max_len:
                seq = seq + [0] * (max_len - len(seq))
            else:
                seq = seq[:max_len]
            self.sequences.append(seq)
            
    def __len__(self):
        return len(self.labels)
        
    def __getitem__(self, idx):
        return torch.tensor(self.sequences[idx], dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.float)

train_dataset = ReviewDataset(df_train, vocab, max_len_lstm)
val_dataset = ReviewDataset(df_val, vocab, max_len_lstm)
test_dataset = ReviewDataset(df_test, vocab, max_len_lstm)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


In [ ]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim=100, hidden_dim=128, output_dim=1, n_layers=2, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=n_layers, bidirectional=True, batch_first=True, dropout=dropout if n_layers > 1 else 0)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, text):
        embedded = self.dropout(self.embedding(text))
        output, (hidden, cell) = self.lstm(embedded)
        pooled = torch.mean(output, dim=1)
        return self.fc(self.dropout(pooled))


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("LSTM Device:", device)

lstm_model = BiLSTMClassifier(vocab_size=len(vocab)).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(lstm_model.parameters(), lr=1e-3)

epochs_lstm = 5
best_val_loss = float('inf')
t0 = time.time()

for epoch in range(epochs_lstm):
    lstm_model.train()
    epoch_loss = 0
    correct = 0
    total = 0
    for seqs, labels in train_loader:
        seqs, labels = seqs.to(device), labels.to(device)
        optimizer.zero_grad()
        predictions = lstm_model(seqs).squeeze(1)
        loss = criterion(predictions, labels)
        
        preds = torch.round(torch.sigmoid(predictions))
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        
    train_loss = epoch_loss / len(train_loader)
    train_acc = correct / total
    
    lstm_model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for seqs, labels in val_loader:
            seqs, labels = seqs.to(device), labels.to(device)
            predictions = lstm_model(seqs).squeeze(1)
            loss = criterion(predictions, labels)
            preds = torch.round(torch.sigmoid(predictions))
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)
            val_loss += loss.item()
            
    valid_loss = val_loss / len(val_loader)
    valid_acc = val_correct / val_total
    print(f"Epoch {epoch+1} | Train Loss: {train_loss:.3f} Acc: {train_acc*100:.2f}% | Val Loss: {valid_loss:.3f} Acc: {valid_acc*100:.2f}%")
    
    if valid_loss < best_val_loss:
        best_val_loss = valid_loss
        torch.save(lstm_model.state_dict(), os.path.join(OUTPUT_PATH, 'best_lstm.pt'))
        
lstm_time = time.time() - t0


In [ ]:
# Đánh giá Bi-LSTM
lstm_model.load_state_dict(torch.load(os.path.join(OUTPUT_PATH, 'best_lstm.pt'), map_location=device))
lstm_model.eval()

test_preds = []
with torch.no_grad():
    for seqs, _ in test_loader:
        seqs = seqs.to(device)
        predictions = lstm_model(seqs).squeeze(1)
        preds = torch.round(torch.sigmoid(predictions))
        test_preds.extend(preds.cpu().numpy())
        
test_preds = np.array(test_preds)
lstm_acc = accuracy_score(y_test, test_preds)
lstm_f1 = f1_score(y_test, test_preds, average='macro')
print(f"Bi-LSTM -> Accuracy: {lstm_acc*100:.2f}%, F1-Score: {lstm_f1*100:.2f}%")


## 5. Các Mô Hình Transformer (PhoBERT & DistilBERT)

Thử nghiệm 3 phương án để xử lý giới hạn phần cứng và độ trễ train:
1. **PhoBERT Fine-tuned (10% Dữ liệu):** Để giảm tải CPU/GPU.
2. **DistilBERT Fine-tuned (10% Dữ liệu):** Rút gọn, nhanh và mượt hơn.
3. **PhoBERT Pre-trained Full (0s Train):** Chạy trực tiếp từ model đã được train hoàn chỉnh trên tập lớn.

In [ ]:
class TransformerDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
        
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'targets': torch.tensor(label, dtype=torch.long)
        }


In [ ]:
# Tải nhanh tập subsample 10% cho Transformer trên local/Colab CPU
df_train_sub = df_train.sample(frac=0.1, random_state=42)
df_val_sub = df_val.sample(frac=0.1, random_state=42)
df_test_sub = df_test.sample(frac=0.1, random_state=42)

# --- 5.1. HUẤN LUYỆN PHO-BERT FINE-TUNED ---
print("Đang tải PhoBERT và tokenizer...")
phobert_tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base", use_fast=False)
phobert_model = AutoModelForSequenceClassification.from_pretrained("vinai/phobert-base", num_labels=2).to(device)

train_ds_pb = TransformerDataset(df_train_sub['clean_comment'].values, df_train_sub['sentiment'].values, phobert_tokenizer)
test_ds_pb = TransformerDataset(df_test_sub['clean_comment'].values, df_test_sub['sentiment'].values, phobert_tokenizer)

train_loader_pb = DataLoader(train_ds_pb, batch_size=16, shuffle=True)
test_loader_pb = DataLoader(test_ds_pb, batch_size=16, shuffle=False)

optimizer_pb = optim.AdamW(phobert_model.parameters(), lr=2e-5)
t0 = time.time()

print("Đang huấn luyện PhoBERT (1 epoch demo)...")
phobert_model.train()
for batch in train_loader_pb:
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    targets = batch['targets'].to(device)
    
    optimizer_pb.zero_grad()
    outputs = phobert_model(input_ids=input_ids, attention_mask=attention_mask, labels=targets)
    loss = outputs.loss
    loss.backward()
    optimizer_pb.step()
    
phobert_time = time.time() - t0
torch.save(phobert_model.state_dict(), os.path.join(OUTPUT_PATH, 'best_phobert.pt'))

# Đánh giá
phobert_model.eval()
pb_preds = []
with torch.no_grad():
    for batch in test_loader_pb:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = phobert_model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)
        pb_preds.extend(preds.cpu().numpy())

phobert_acc = accuracy_score(df_test_sub['sentiment'].values, pb_preds)
phobert_f1 = f1_score(df_test_sub['sentiment'].values, pb_preds, average='macro')
print(f"PhoBERT Fine-tuned -> Acc: {phobert_acc*100:.2f}%, F1: {phobert_f1*100:.2f}%")


In [ ]:
# --- 5.2. HUẤN LUYỆN DISTILBERT FINE-TUNED (Nhanh hơn) ---
print("Đang tải DistilBERT...")
distil_tokenizer = AutoTokenizer.from_pretrained("distilbert-base-multilingual-cased")
distil_model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-multilingual-cased", num_labels=2).to(device)

train_ds_db = TransformerDataset(df_train_sub['clean_comment'].values, df_train_sub['sentiment'].values, distil_tokenizer)
test_ds_db = TransformerDataset(df_test_sub['clean_comment'].values, df_test_sub['sentiment'].values, distil_tokenizer)
train_loader_db = DataLoader(train_ds_db, batch_size=16, shuffle=True)
test_loader_db = DataLoader(test_ds_db, batch_size=16, shuffle=False)

optimizer_db = optim.AdamW(distil_model.parameters(), lr=3e-5)
t0 = time.time()

print("Đang huấn luyện DistilBERT (1 epoch demo)...")
distil_model.train()
for batch in train_loader_db:
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    targets = batch['targets'].to(device)
    
    optimizer_db.zero_grad()
    outputs = distil_model(input_ids=input_ids, attention_mask=attention_mask, labels=targets)
    loss = outputs.loss
    loss.backward()
    optimizer_db.step()
    
distil_time = time.time() - t0
torch.save(distil_model.state_dict(), os.path.join(OUTPUT_PATH, 'best_distilbert.pt'))

# Đánh giá
distil_model.eval()
db_preds = []
with torch.no_grad():
    for batch in test_loader_db:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = distil_model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)
        db_preds.extend(preds.cpu().numpy())

distil_acc = accuracy_score(df_test_sub['sentiment'].values, db_preds)
distil_f1 = f1_score(df_test_sub['sentiment'].values, db_preds, average='macro')
print(f"DistilBERT Fine-tuned -> Acc: {distil_acc*100:.2f}%, F1: {distil_f1*100:.2f}%")


In [ ]:
# --- 5.3. ĐÁNH GIÁ PHO-BERT PRE-TRAINED SENTIMENT (Không cần train) ---
print("Đang tải mô hình PhoBERT đã được huấn luyện sẵn Sentiment (wonrax)...")
pre_tokenizer = AutoTokenizer.from_pretrained("wonrax/phobert-base-vietnamese-sentiment", use_fast=False)
pre_model = AutoModelForSequenceClassification.from_pretrained("wonrax/phobert-base-vietnamese-sentiment").to(device)
pre_model.eval()

pre_preds = []
y_test_sub = df_test_sub['sentiment'].values
t0 = time.time()

with torch.no_grad():
    for text in df_test_sub['clean_comment'].values:
        inputs = pre_tokenizer(text, return_tensors="pt", max_length=128, padding='max_length', truncation=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        outputs = pre_model(**inputs)
        
        # Nhãn gốc: 0: NEG, 1: NEU, 2: POS. Ánh xạ nhị phân: POS > NEG -> 1 (Tích cực), ngược lại -> 0 (Tiêu cực)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)[0]
        neg_prob = probs[0].item()
        pos_prob = probs[2].item()
        pred = 1 if pos_prob > neg_prob else 0
        pre_preds.append(pred)
        
pre_time = time.time() - t0
pre_acc = accuracy_score(y_test_sub, pre_preds)
pre_f1 = f1_score(y_test_sub, y_test_sub, average='macro')
print(f"PhoBERT Pre-trained Full (0s train) -> Acc: {pre_acc*100:.2f}%, F1: {pre_f1*100:.2f}%")


## 6. So Sánh Kết Quả Tổng Hợp

In [ ]:
summary_data = {
    'Mô hình': [
        'Naive Bayes (Baseline)', 
        'Logistic Regression (Baseline)', 
        'PyTorch Bi-LSTM (Advanced)',
        'PhoBERT (10% Fine-tuned)',
        'DistilBERT (10% Fine-tuned)',
        'PhoBERT (Pre-trained Full - 0s Train)'
    ],
    'Accuracy (%)': [nb_acc*100, lr_acc*100, lstm_acc*100, phobert_acc*100, distil_acc*100, pre_acc*100],
    'F1-Score (%)': [nb_f1*100, lr_f1*100, lstm_f1*100, phobert_f1*100, distil_f1*100, pre_f1*100],
    'Thời gian (s)': [nb_time, lr_time, lstm_time, phobert_time, distil_time, pre_time]
}
df_summary = pd.DataFrame(summary_data)
print(df_summary.to_string(index=False))


## 7. Kiểm Thử Tương Tác Cảm Xúc (Interactive Sentiment Tester)

In [ ]:
def test_review(review_text):
    cleaned = preprocess_review(review_text)
    print(f"Review gốc: {review_text}")
    print(f"Sau tiền xử lý: {cleaned}\n")
    
    # 1. Logistic Regression (TF-IDF)
    tfidf_vec = vectorizer.transform([cleaned])
    lr_pred = lr_model.predict(tfidf_vec)[0]
    lr_prob = lr_model.predict_proba(tfidf_vec)[0]
    lr_label = "Tích cực" if lr_pred == 1 else "Tiêu cực"
    lr_conf = lr_prob[1] if lr_pred == 1 else lr_prob[0]
    
    # 2. PyTorch Bi-LSTM
    seq = [vocab.get(w, vocab['<UNK>']) for w in cleaned.split()]
    if len(seq) < max_len_lstm:
        seq = seq + [0] * (max_len_lstm - len(seq))
    else:
        seq = seq[:max_len_lstm]
    lstm_model.eval()
    with torch.no_grad():
        t_seq = torch.tensor([seq], dtype=torch.long).to(device)
        logit = lstm_model(t_seq).squeeze(1).item()
        prob = torch.sigmoid(torch.tensor(logit)).item()
        lstm_label = "Tích cực" if prob >= 0.5 else "Tiêu cực"
        lstm_conf = prob if prob >= 0.5 else (1.0 - prob)
        
    # 3. PhoBERT (10% Fine-tuned)
    phobert_model.eval()
    inputs_pb = phobert_tokenizer(cleaned, return_tensors="pt", max_length=128, padding='max_length', truncation=True)
    inputs_pb = {k: v.to(device) for k, v in inputs_pb.items()}
    with torch.no_grad():
        outputs_pb = phobert_model(**inputs_pb)
        probs_pb = torch.nn.functional.softmax(outputs_pb.logits, dim=1)
        pb_conf, pb_pred = torch.max(probs_pb, dim=1)
        pb_label = "Tích cực" if pb_pred.item() == 1 else "Tiêu cực"
        pb_conf = pb_conf.item()
        
    # 4. PhoBERT (Pre-trained Full - wonrax)
    pre_model.eval()
    inputs_pre = pre_tokenizer(cleaned, return_tensors="pt", max_length=128, padding='max_length', truncation=True)
    inputs_pre = {k: v.to(device) for k, v in inputs_pre.items()}
    with torch.no_grad():
        outputs_pre = pre_model(**inputs_pre)
        probs_pre = torch.nn.functional.softmax(outputs_pre.logits, dim=-1)[0]
        neg_p = probs_pre[0].item()
        pos_p = probs_pre[2].item()
        pre_label = "Tích cực" if pos_p > neg_p else "Tiêu cực"
        pre_conf = max(pos_p, neg_p) / (pos_p + neg_p + 1e-9)
        
    print("-"*70)
    print(f"Logistic Regression (Baseline):      {lr_label:<10} (Độ tin cậy: {lr_conf*100:.2f}%)")
    print(f"PyTorch Bi-LSTM (Deep Learning):     {lstm_label:<10} (Độ tin cậy: {lstm_conf*100:.2f}%)")
    print(f"PhoBERT (10% Fine-tuned):           {pb_label:<10} (Độ tin cậy: {pb_conf*100:.2f}%)")
    print(f"PhoBERT (Pre-trained Full):          {pre_label:<10} (Độ tin cậy: {pre_conf*100:.2f}%)")
    print("-"*70)
    print()

# Thử nghiệm một vài câu cụ thể
test_review("Sản phẩm quá đẹp, giao hàng siêu nhanh, đóng gói cẩn thận 10 điểm")
test_review("Mua về chưa xài đã hỏng, shop làm ăn tắc trách quá, không bao giờ mua lại")
test_review("Sản phẩm dùng cũng tạm được, không quá xuất sắc nhưng rẻ")
